In [ ]:
import gc
import os
import random
import sys

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import xarray as xr
from keras import Model
from keras.layers import Dense, Input, Dropout, Conv2D, Flatten, MaxPooling2D

import data_processing
import file_methods
import plots

In [ ]:
def get_settings(experiment_name):
    experiments = {
        "CanESM5_rsut": {
            "input_region": "X_train.nc",
            "detrend": True,
            "datafolder": "../../data/splits/",
            "anomalies": True,
            "anomalies_years": (0, 30),

            "data_period":"_1850-2100",
            "input_var": "tas", 
            "label_var": "cre",
            "n_train_val_test": (17,4,4),
            "yr_bounds": (1850, 2014),

            "network_type": "cnn",
            "kernel_size": 3,
            "kernels": [32, 32],
            "ridge_param": [0.0, 0.0],
            "kernel_act": ["relu", "relu"],
            "hiddens": [32, 16],
            "act_fun": ["elu", "elu"],
            "learning_rate": 0.000005,
            "batch_size": 32,
            "rng_seed": 42,
            "rng_seed_list": np.arange(5).tolist(),
            "n_epochs": 25_000,
        },
        "CanESM5_rlut": {
            "input_region": "X_train.nc",
            "detrend": True,
            "datafolder": "../../data/splits/",
            "anomalies": True,
            "anomalies_years": (0, 30),

            "data_period":"_1850-2100",
            "input_var": "tas", 
            "label_var": "cre",
            "n_train_val_test": (17,4,4),
            "yr_bounds": (1850, 2014),

            "network_type": "cnn",
            "kernel_size": 3,
            "kernels": [32, 32],
            "ridge_param": [0.0, 0.0],
            "kernel_act": ["relu", "relu"],
            "hiddens": [32, 16],
            "act_fun": ["elu", "elu"],
            "learning_rate": 0.000005,
            "batch_size": 32,
            "rng_seed": 42,
            "rng_seed_list": np.arange(5).tolist(),
            "n_epochs": 25_000,
        },
    }
    exp_dict = experiments[experiment_name]
    exp_dict['exp_name'] = experiment_name
    if 'n_train_val_test' in exp_dict.keys():
        exp_dict['n_members'] = int(np.sum(exp_dict["n_train_val_test"]))

    return exp_dict



In [ ]:
def compile_model(x_train, y_train, settings):
    # NORMALIZATION LAYER
    inputs = Input(shape=x_train.shape[1:])
    layers = inputs

    # DEFINE THE MODEL ARCHITECTURE
    # feed-forward network
    if settings["network_type"] == "ffn":
        assert len(settings["hiddens"]) == len(settings["act_fun"]) == len(settings["ridge_param"])

        layers = Flatten()(layers)
        layers = Dropout(rate=settings["dropout_rate"][0], seed=settings["rng_seed"])(layers)

        for hidden, activation, ridge in zip(settings["hiddens"], settings["act_fun"], settings["ridge_param"]):
            layers = Dense(
                hidden,
                activation=activation,
                use_bias=True,
                kernel_regularizer=tf.keras.regularizers.l1_l2(l1=0.00, l2=ridge),
                bias_initializer=tf.keras.initializers.RandomNormal(seed=settings["rng_seed"]),
                kernel_initializer=tf.keras.initializers.RandomNormal(seed=settings["rng_seed"]),
            )(layers)

    # convolutional neural network
    elif settings["network_type"] == "cnn":
        assert len(settings["kernels"]) == len(settings["kernel_act"])
        assert len(settings["hiddens"]) == len(settings["act_fun"])

        for kernel, kernel_act in zip(settings["kernels"], settings["kernel_act"]):
            layers = Conv2D(
                kernel,
                (settings["kernel_size"], settings["kernel_size"]),
                use_bias=True,
                activation=kernel_act,
                padding="same",
                bias_initializer=tf.keras.initializers.RandomNormal(seed=settings["rng_seed"]),
                kernel_initializer=tf.keras.initializers.RandomNormal(seed=settings["rng_seed"]),
            )(layers)
            layers = MaxPooling2D((2, 2))(layers)

        # make final dense layers
        layers = Flatten()(layers)
        for hidden, activation in zip(settings["hiddens"], settings["act_fun"]):
            layers = Dense(
                hidden,
                activation=activation,
                use_bias=True,
                bias_initializer=tf.keras.initializers.RandomNormal(seed=settings["rng_seed"]),
                kernel_initializer=tf.keras.initializers.RandomNormal(seed=settings["rng_seed"]),
            )(layers)

    else:
        raise NotImplementedError()

    # DEFINE THE OUTPUT LAYER
    output_layer = Dense(
        y_train.shape[-1],
        activation="linear",
        use_bias=True,
        bias_initializer=tf.keras.initializers.RandomNormal(seed=settings["rng_seed"]),
        kernel_initializer=tf.keras.initializers.RandomNormal(seed=settings["rng_seed"]),
    )(layers)

    # CONSTRUCT THE MODEL
    model = Model(inputs, output_layer)

    # COMPILE THE MODEL
    model.compile(
        optimizer=tf.keras.optimizers.legacy.Adam(learning_rate=settings["learning_rate"]), \
        loss=tf.keras.losses.MeanSquaredError(), \
        metrics=["mae", "mse"] \
    )

    return model

In [ ]:
"""Metrics for generic plotting.
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.feature as cfeature
from shapely.geometry.polygon import LinearRing
import cartopy.crs as ccrs

import cartopy as ct
import warnings
import sklearn
from sklearn import metrics

# Matplotlib colors
npcols = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf', 
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf', 
]

warnings.filterwarnings("ignore")
mpl.rcParams["figure.facecolor"] = "white"
mpl.rcParams["figure.dpi"] = 150

data_crs = ccrs.PlateCarree()
proj_Global = ccrs.EqualEarth(central_longitude=200)

def savefig(filename, dpi=300):
    for fig_format in (".png", ".pdf"):
        plt.savefig(filename + fig_format,
                    bbox_inches="tight",
                    dpi=dpi)


def round_down_ten(x):
    y = np.floor(x / 10.) * 10.
    return y


def plot_metrics(history, metric):
    imin = np.argmin(history.history['val_loss'])

    plt.plot(history.history[metric], label='training')
    plt.plot(history.history['val_' + metric], label='validation')
    plt.title(metric)
    plt.axvline(x=imin, linewidth=.5, color='gray', alpha=.5)
    plt.legend()


def plot_metrics_panels(history, predictions):
    # BASELINES TO BEAT
    baseline_mae = np.mean(np.abs(predictions["labels_test"] - 0.0))
    baseline_mse = np.mean((predictions["labels_test"] - 0.0)**2)

    # DO THE PLOTTING
    plt.subplots(figsize=(20, 4))

    plt.subplot(1, 4, 1)
    plot_metrics(history, 'loss')
    plt.ylim(0, .5)
    plt.axhline(y=baseline_mse, color="gray", linestyle="--", label="baseline")

    plt.subplot(1, 4, 2)
    plot_metrics(history, "mae")
    plt.ylim(0, .5)
    plt.axhline(y=baseline_mae, color="gray", linestyle="--", label="baseline")

    plt.legend()


def plot_pred_vs_truth(predictions, settings, ms=5, label=''):

    mse = sklearn.metrics.mean_squared_error(predictions["labels_val"], predictions["pred_val"])
    if np.isnan(mse):
        mse = 0.
    plt.plot(predictions["labels_val"], predictions["pred_val"], '.', alpha=.5,
                label=label + ' (MSE=' + str(mse.round(3)) + ')', markersize=ms)

    plt.plot((-10, 10), (-10, 10), '--k', alpha=.5)
    plt.ylim(-8, 3)
    plt.xlim(-8, 3)
    plt.ylabel('predicted')
    plt.xlabel('truth')
    plt.legend(frameon=False, fontsize=8)
    plt.gca().set_aspect('equal')


def setup_figure(nCols=1,nRows=1,size=(15,15),mask=True):

    map_proj = proj_Global

    land_feature = cfeature.NaturalEarthFeature(
        category='physical',
        name='land',
        scale='50m',
        facecolor='gray',
        edgecolor='k',
        linewidth=.25,
    )

    fig = plt.figure(figsize=size)
    if nCols==1 and nRows==1:
        ax = fig.add_subplot(1, 1, 1, projection=map_proj)

        ax.coastlines('50m', linewidth=0.8)
        ax.tick_params(axis='both', which='major', labelsize=10)
        ax.gridlines(
            draw_labels=False, linewidth=0.5, color='gray', alpha=0.5, linestyle='--'
        )

        if mask:
            ax.add_feature(land_feature)
    elif nCols > 1 and nRows > 1:
        ax = np.empty((nCols,nRows),dtype=object)
        iF = 1
        for jj in range(nRows):
            for ii in range(nCols):
                ax[ii,jj] = fig.add_subplot(nRows, nCols, iF, projection=map_proj)

                ax[ii,jj].coastlines('50m', linewidth=0.8)
                ax[ii,jj].tick_params(axis='both', which='major', labelsize=10)
                ax[ii,jj].gridlines(
                    draw_labels=False, linewidth=0.5, color='gray', alpha=0.5, linestyle='--'
                )
                if mask:
                    ax[ii,jj].add_feature(land_feature)
                iF += 1
    elif nCols > 1:
        ax = np.empty((nCols),dtype=object)
        for ii in range(nCols):
            ax[ii] = fig.add_subplot(1, nCols, ii+1, projection=map_proj)

            ax[ii].coastlines('50m', linewidth=0.8)
            ax[ii].tick_params(axis='both', which='major', labelsize=10)
            ax[ii].gridlines(
                draw_labels=False, linewidth=0.5, color='gray', alpha=0.5, linestyle='--'
            )
            if mask:
                ax[ii].add_feature(land_feature)
    elif nRows > 1:
        ax = np.empty((nRows),dtype=object)
        for ii in range(nRows):
            ax[ii] = fig.add_subplot(nRows, 1, ii+1, projection=map_proj)

            ax[ii].coastlines('50m', linewidth=0.8)
            ax[ii].tick_params(axis='both', which='major', labelsize=10)
            ax[ii].gridlines(
                draw_labels=False, linewidth=0.5, color='gray', alpha=0.5, linestyle='--'
            )
            if mask:
                ax[ii].add_feature(land_feature)

            
    return fig, ax


def add_mask(ax,region,lon=None,lat=None):
    
    if region == "land":
        land_feature = cfeature.NaturalEarthFeature(
            category='physical',
            name='land',
            scale='50m',
            facecolor='gray',
            edgecolor='k',
            linewidth=.25,
        )
        ax.add_feature(land_feature)
    elif region[:4] == "reg_":

        regions_range_dict = regions.get_region_dict(region)

        min_lon, max_lon = regions_range_dict["lon_range"]
        min_lat, max_lat = regions_range_dict["lat_range"]
        region = [min_lon, min_lat, max_lon, max_lat]

        add_square(ax,region,data_crs,facecolor='gray')

    elif region[-3:] == ".nc":

        mask = xr.load_dataarray(SHAPE_DIRECTORY + region).to_numpy()

        mask[mask>0.5] = np.nan
        mask_cyc, lons_cyc = add_cyclic_point(mask, coord=lon)
        ax.pcolormesh(lons_cyc,lat,mask_cyc,cmap="gray")

    return ax


def add_square(ax,region,crs,**kwargs):
    # Region = lon1, lat1; lon2, lat2

    lons = [region[0],region[2],region[2],region[0]]
    lats = [region[1],region[1],region[3],region[3]]
    ring = LinearRing(list(zip(lons,lats)))

    ax.add_geometries([ring],crs,**kwargs)

    return ax


def add_loc_square(ax,settings,**kwargs):
    if "mask_region" in settings.keys():
        maskout=settings["mask_region"]
    else:
        return ax

    if maskout == "indonesia":
        min_lon, max_lon = 75., 130.
        min_lat, max_lat = -15., 10.
    elif maskout == "westpacific":
        min_lon, max_lon = 170., 230.
        min_lat, max_lat = -30., -5.
    elif maskout == "eastpacific":
        min_lon, max_lon = 210., 280.
        min_lat, max_lat = -10., 10.
    elif maskout == "caribbean":
        min_lon, max_lon = 250., 320.
        min_lat, max_lat = 5., 25.
    elif maskout == "brazil":
        min_lon, max_lon = 310., 350.
        min_lat, max_lat = -25., -5.
    elif maskout == "namibia":
        min_lon, max_lon = 0., 20.
        min_lat, max_lat = -45., -10.
    else:
        raise NotImplementedError("no such mask type.")
    
    if min_lon > 180 and max_lon > 180:
        min_lon = min_lon - 360
        max_lon = max_lon - 360
    
    add_square(ax,[min_lon,min_lat,max_lon,max_lat],data_crs,**kwargs)

    return ax

def round_to_n(x,n):
    if x == 0:
        return x
    else:
        return np.round(x, -int(np.floor(np.log10(abs(x)))) + (n - 1))
    
def num_lab(x,n):
    return str(round_to_n(x,n))

def get_area(lat,lon,mask=None):
    _, Y = np.meshgrid(lon,lat)

    dy = np.deg2rad(np.diff(lat)[0])
    dx = np.deg2rad(np.diff(lon)[0]) * np.cos(np.deg2rad(Y))

    area = dy * dx

    if mask is not None:
        area = area * mask
        area[np.isnan(area)] = 0.

    return area/np.sum(area)

In [ ]:
"""Functions for working with generic files.

Functions
---------
get_model_name(settings)
get_netcdf_da(filename)
save_pred_obs(pred_vector, filename)
save_tf_model(model, model_name, directory, settings)
get_cmip_filenames(settings, verbose=0)
"""

import xarray as xr
import json
import pickle
import tensorflow as tf


def get_model_name(settings, suffix=""):
    model_name = (settings["exp_name"] + suffix + '_rngseed' + str(settings["rng_seed"]))
    return model_name


def get_netcdf_da(filename, members, settings):
    print(filename)
    try:
        return xr.open_dataarray(filename)[members, :, :, :]
    except:
        return xr.open_dataarray(filename)[members, :]


def save_predictions(predictions, filename):
    with open(filename, 'wb') as f:
        pickle.dump(predictions, f)


def load_predictions(filename):
    with open(filename, 'rb') as f:
        predictions = pickle.load(f)
    return predictions


def load_tf_model(model_name, directory):
    # loading a tf model
    model = tf.keras.models.load_model(
        directory + model_name + "_model",
        compile=False,
    )
    return model


def save_tf_model(model, model_name, directory, settings):
    # save the tf model
    tf.keras.models.save_model(model, directory + model_name + "_model", overwrite=True)

    # save the meta data
    with open(directory + model_name + '_metadata.json', 'w') as json_file:
        json_file.write(json.dumps(settings))


def get_simulations(var, directory, members, settings):

    if settings["detrend"] is True:
        v = get_netcdf_da(directory + settings["datafolder"] + var \
                          + settings["data_period"] + "_detrend.nc", members, settings)
    elif settings["detrend"] == "1pctCO2":
        v = get_netcdf_da(directory + settings["datafolder"] + var \
                          + "_150.nc", members, settings)
    else:
        v = get_netcdf_da(directory + settings["datafolder"] + var \
                          + settings["data_period"] + ".nc", members, settings)
    return v


In [ ]:

plt.style.use("default")
mpl.rcParams["savefig.facecolor"] = "white"
mpl.rcParams["figure.dpi"] = 150
savefig_dpi = 300
# tf.config.set_visible_devices([], "GPU")  # turn-off tensorflow-metal if it is on

print(f"python version = {sys.version}")
print(f"numpy version = {np.__version__}")
print(f"xarray version = {xr.__version__}")
print(f"tensorflow version = {tf.__version__}")
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

# ---------------------------------------------------------
# ---------------------------------------------------------

EXP_NAME_LIST = ( \
    "CanESM5_rsut",
    "CanESM5_rlut",
)
OVERWRITE_MODEL = False

from DIRECTORIES import MODEL_DIRECTORY, PREDICTIONS_DIRECTORY, DATA_DIRECTORY, DIAGNOSTICS_DIRECTORY, FIGURES_DIRECTORY

# ---------------------------------------------------------
# ---------------------------------------------------------

for EXP_NAME in EXP_NAME_LIST:
    print("------" + EXP_NAME + "------")
    settings = get_settings(EXP_NAME)
    # display(settings)

    for rng_seed in settings["rng_seed_list"]:
        settings["rng_seed"] = rng_seed
        tf.random.set_seed(settings["rng_seed"])
        random.seed(settings["rng_seed"])
        np.random.seed(settings["rng_seed"])

        # GET MODEL NAME AND CHECK IF IT EXISTS
        model_name = file_methods.get_model_name(settings)
        if os.path.exists(MODEL_DIRECTORY + model_name + "_model") and OVERWRITE_MODEL is False:
            print("\n" + model_name + "exists. Skipping...")
            print("================================\n")
            continue

            # GET THE DATA
        (
            x_train,
            x_val,
            x_test,
            labels_train,
            labels_val,
            labels_test,
            lat,
            lon,
            map_shape,
            member_shape,
            time_shape,
            norm_dict,
            member_enrollment,
            settings,
        ) = data_processing.get_cmip_data(
            DATA_DIRECTORY,
            settings,
            n_train_val_test=settings["n_train_val_test"],
        )

        # ----------------------------------------
        tf.keras.backend.clear_session()
        # define early stopping callback (cannot be done elsewhere)
        early_stopping = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=10, verbose=1, mode="auto", restore_best_weights=True
        )

        model = compile_model(x_train, labels_train, settings)
        model.summary()

        history = model.fit(
            x_train,
            labels_train,
            epochs=settings["n_epochs"],
            batch_size=settings["batch_size"],
            shuffle=True,
            validation_data=[x_val, labels_val],
            callbacks=[
                early_stopping,
            ],
            verbose=2,
        )

        # make predictions dictionary
        predictions = {
            "labels_train": labels_train,
            "pred_train": model.predict(x_train),
            "labels_val": labels_val,
            "pred_val": model.predict(x_val),
            "labels_test": labels_test,
            "pred_test": model.predict(x_test),
        }

        # clean-up from model.predict
        _ = gc.collect()

        # ----------------------------------------
        # save the tensorflow model and predictions
        file_methods.save_tf_model(model, model_name, MODEL_DIRECTORY, settings)
        file_methods.save_predictions(predictions, PREDICTIONS_DIRECTORY + model_name + "_predictions.pickle")

        # #----------------------------------------
        # create and save diagnostics plots
        plots.plot_metrics_panels(history, predictions)
        plt.savefig(
            DIAGNOSTICS_DIRECTORY + model_name + "_metrics_diagnostic" + ".png", dpi=savefig_dpi, bbox_inches="tight"
        )
        plt.close()

        plots.plot_pred_vs_truth(predictions, settings)
        plt.savefig(FIGURES_DIRECTORY + model_name + "_pred_vs_truth" + ".png", dpi=savefig_dpi, bbox_inches="tight")
        plt.close()

        # #----------------------------------------
        # save predictions over all members to use as labels for other networks
        (x_all, __, __, labels_all, __, __, __, __, __, member_shape_all, __, __, __, settings) = data_processing.get_cmip_data(
            DATA_DIRECTORY,
            settings,
            n_train_val_test=(settings["n_members"], 0, 0),
        )

        pred_all = model.predict(x_all)
        pred_all = pred_all.reshape(member_shape_all, time_shape, 1)
        labels_all = labels_all.reshape(member_shape_all, time_shape, 1)

        predictions_all = {"pred_all": pred_all, "labels_all": labels_all}
        file_methods.save_predictions(predictions_all, PREDICTIONS_DIRECTORY + model_name + "_all_predictions.pickle")

        # clean-up from model.predict
        _ = gc.collect()
    print("------" + EXP_NAME + " DONE ------")